# 04 — AI Architecture for Physics Simulation

**Adaptive Multiscale Biological Simulation AI**

Now that we have a working physics engine (LJ fluid, thermostats, observables), we layer AI on top.
This notebook covers three pillars:

1. **GNNs (Graph Neural Networks)** — learn force fields
2. **PINNs (Physics-Informed Neural Networks)** — solve PDEs from scratch
3. **Neural Operators** — learn the solution operator (Fourier Neural Operator)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 13, 'figure.dpi': 150})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Part 1: GNN for Molecular Force Prediction

We build a message-passing neural network (MPNN) that learns the LJ force field.

$$h_i^{(t+1)} = \text{MLP}\left(h_i^{(t)} \,\|\, \sum_{j \in \mathcal{N}(i)} \phi(h_i^{(t)}, h_j^{(t)}, r_{ij}, r_{ij}^2)\right)$$

In [ ]:
class GNNForceField(nn.Module):
    """
    Message-passing GNN for molecular force prediction.
    
    Node features: atomic number (one-hot encoded)
    Edge features: distance r and distance squared r²
    Output: per-node 3D forces + per-atom energy contributions
    """
    
    def __init__(self, n_features=32, n_layers=4, hidden_dim=64):
        super().__init__()
        self.n_layers = n_layers
        
        self.node_embed = nn.Linear(n_features, hidden_dim)
        
        self.message_layers = nn.ModuleList()
        self.update_layers = nn.ModuleList()
        for _ in range(n_layers):
            self.message_layers.append(nn.Linear(hidden_dim * 2 + 2, hidden_dim))
            self.update_layers.append(nn.Linear(hidden_dim * 2, hidden_dim))
        
        self.force_head = nn.Linear(hidden_dim, 3)
        self.energy_head = nn.Linear(hidden_dim, 1)
        self.act = F.elu
    
    def forward(self, node_features, edge_index, edge_r, edge_r2):
        h = self.act(self.node_embed(node_features))  # (n, hidden)
        
        src, tgt = edge_index
        
        for layer in range(self.n_layers):
            h_i, h_j = h[src], h[tgt]
            edge_features = torch.cat([h_i, h_j, edge_r2.unsqueeze(1), edge_r.unsqueeze(1)], dim=-1)
            
            messages = self.act(self.message_layers[layer](edge_features))
            
            agg = torch.zeros_like(h)
            agg.index_add_(0, tgt, messages)
            
            node_update = torch.cat([h, agg], dim=-1)
            h = self.act(h + self.act(self.update_layers[layer](node_update)))
        
        forces = self.force_head(h)
        energy_atomic = self.energy_head(h).squeeze(-1)
        
        return forces, energy_atomic

# Verify the model
model = GNNForceField(n_features=4, n_layers=4, hidden_dim=64)
n_example = 10
dummy_nodes = torch.randn(n_example, 4)

edges, edge_r_list, edge_r2_list = [], [], []
for i in range(n_example):
    for j in range(n_example):
        if i != j:
            r = torch.norm(torch.randn(3)) + 1.0
            edges.append([i, j])
            edge_r_list.append(r)
            edge_r2_list.append(torch.tensor(r * r))
edge_index = torch.tensor(edges, dtype=torch.long).t()
edge_r = torch.stack(edge_r_list)
edge_r2 = torch.stack(edge_r2_list)

forces_out, energy_out = model(dummy_nodes, edge_index, edge_r, edge_r2)
print(f"Forces shape: {forces_out.shape} (expected ({n_example}, 3))")
print(f"Energy shape: {energy_out.shape} (expected ({n_example},))")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training the GNN on LJ Data

Generate synthetic LJ data as training set. The network learns to approximate $F_{LJ}(r)$ and $V_{LJ}(r)$.

In [ ]:
from src.core.potentials import lj_force, lj_potential, SIGMA, EPSILON

def generate_lj_dataset(n_configs=500, n_atoms_range=(2, 8), n_edges_per_config=100):
    """Generate synthetic LJ dataset for GNN training."""
    X_positions = []
    X_forces = []
    X_energies = []
    X_edges_r = []
    X_edges_r2 = []
    
    for c in range(n_configs):
        n = np.random.randint(n_atoms_range[0], n_atoms_range[1])
        box = np.random.uniform(3.0, 10.0)
        
        # Random positions in box
        positions = np.random.uniform(0, box, (n, 3))
        
        # Gaussian perturbations for variety
        positions += np.random.randn(n, 3) * 0.5
        
        # Compute LJ forces and energies
        for i in range(n):
            fx, fy, fz = 0.0, 0.0, 0.0
            Ei = 0.0
            for j in range(n):
                if i == j:
                    continue
                dr = positions[i] - positions[j]
                r2 = np.dot(dr, dr)
                if r2 > 0 and r2 < 6.25:  # RCUT=2.5
                    f = lj_force(r2)
                    vx = lj_potential(r2)
                    fx += f * dr[0]
                    fy += f * dr[1]
                    fz += f * dr[2]
                    Ei += vx
            X_forces.append([fx, fy, fz])
            X_energies.append(Ei)
        
        # Collect edges
        for i in range(n):
            for j in range(n):
                if i != j:
                    dr = positions[i] - positions[j]
                    r2 = np.dot(dr, dr)
                    if r2 > 0 and r2 < 6.25:
                        r = np.sqrt(r2)
                        X_edges_r.append(r)
                        X_edges_r2.append(r2)
        
        X_positions.append(positions.flatten())
    
    X_positions = np.array(X_positions)
    X_forces = np.array(X_forces)
    X_energies = np.array(X_energies)
    X_edges_r = np.array(X_edges_r)
    X_edges_r2 = np.array(X_edges_r2)
    
    print(f"Dataset: {X_positions.shape[0]} positions x {X_positions.shape[1]} dims")
    print(f"Forces: {X_forces.shape[0]} samples, shape {X_forces.shape[1:]}")
    print(f"Energies: {X_energies.shape[0]} samples")
    print(f"Edge pairs: {len(X_edges_r)}")
    
    return (torch.tensor(X_positions, dtype=torch.float32),
            torch.tensor(X_forces, dtype=torch.float32),
            torch.tensor(X_energies, dtype=torch.float32),
            torch.tensor(X_edges_r, dtype=torch.float32),
            torch.tensor(X_edges_r2, dtype=torch.float32))

X_pos, X_force_true, X_energy_true, X_edges_r, X_edges_r2 = generate_lj_dataset(n_configs=500)

train_size = int(0.8 * len(X_pos))
train_loader = DataLoader(TensorDataset(X_pos[:train_size], X_force_true[:train_size], X_energy_true[:train_size], X_edges_r, X_edges_r2), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_pos[train_size:], X_force_true[train_size:], X_energy_true[train_size:], X_edges_r, X_edges_r2), batch_size=64)

# Create dummy edge_index for GNN
n_nodes = 8
edge_list = []
for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            edge_list.append([i, j])
dummy_edge_index = torch.tensor(edge_list, dtype=torch.long).t()

## Training Loop

Train the GNN to minimize force-field reconstruction loss.

In [ ]:
model = GNNForceField(n_features=72, n_layers=4, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

n_epochs = 100
loss_history = {'train': [], 'val_force': [], 'val_energy': []}

print(f"Training {n_epochs} epochs...")
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for X_p, X_f, X_e, X_er, X_er2 in train_loader:
        X_p = X_p.to(device)
        X_f = X_f.to(device)
        X_e = X_e.to(device)
        X_er = X_er.to(device)
        X_er2 = X_er2.to(device)
        
        forces_pred, energy_pred = model(X_p, dummy_edge_index.to(device), X_er, X_er2)
        
        force_loss = F.mse_loss(forces_pred, X_f)
        energy_loss = F.mse_loss(energy_pred.sum(), X_e.sum())
        loss = force_loss + 0.1 * energy_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    scheduler.step()
    
    loss_history['train'].append(total_loss / n_batches)
    
    if epoch % 10 == 0:
        model.eval()
        val_loss_f, val_loss_e = 0, 0
        with torch.no_grad():
            for X_p, X_f, X_e, X_er, X_er2 in val_loader:
                X_p, X_f, X_e, X_er, X_er2 = X_p.to(device), X_f.to(device), X_e.to(device), X_er.to(device), X_er2.to(device)
                fp, ep = model(X_p, dummy_edge_index.to(device), X_er, X_er2)
                val_loss_f = F.mse_loss(fp, X_f).item()
                val_loss_e = F.mse_loss(ep.sum(), X_e.sum()).item()
        loss_history['val_force'].append(val_loss_f)
        loss_history['val_energy'].append(val_loss_e)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d}/{n_epochs} | L_force={force_loss.item():.6f} | L_energy={energy_loss.item():.6f}")

print("\nTraining complete!")

## Visualization: GNN Loss Curves + Force Prediction Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
epochs_arr = np.arange(n_epochs)
axes[0].plot(epochs_arr, loss_history['train'], 'b-', linewidth=2, label='Train (combined)')
if loss_history['val_force']:
    axes[0].plot(np.arange(0, n_epochs, 10), loss_history['val_force'], 'ro-', markersize=7, label='Val force MSE')
if loss_history['val_energy']:
    axes[0].plot(np.arange(0, n_epochs, 10), loss_history['val_energy'], 'gs-', markersize=7, label='Val energy MSE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('GNN Force Field Training')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Compare predicted vs true forces on validation set
model.eval()
with torch.no_grad():
    test_batch = list(val_loader)[0]
    X_p_test = test_batch[0].to(device)
    X_e_test = test_batch[2].to(device)
    _, _, _, _, X_er_test = test_batch
    _, _ = model(X_p_test, dummy_edge_index.to(device), X_er_test.to(device), X_er_test.to(device) ** 2)
    # Just compare final loss
    pred_e = model(X_p_test, dummy_edge_index.to(device), X_er_test.to(device), X_er_test.to(device) ** 2)[1]

sample_idx = np.random.choice(len(X_energy_true[train_size:]), 500)
true_ens = X_energy_true[train_size:][sample_idx].numpy()

axes[1].hist(energy_pred[:, 0].cpu().numpy(), bins=40, alpha=0.5, label='GNN Energy (predicted)', density=True)
axes[1].hist(true_ens, bins=40, alpha=0.5, label='LJ Energy (true)', density=True)
axes[1].set_xlabel('Per-atom Energy (ε)')
axes[1].set_ylabel('PDF')
axes[1].set_title('Energy Distribution: GNN vs LJ Ground Truth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figs/04_gnn_force_field.png', bbox_inches='tight')
plt.show()
print("GNN plots saved to figs/04_gnn_force_field.png")

## Part 2: PINNs (Physics-Informed Neural Networks)

PINNs solve PDEs by embedding the physics directly in the loss function.

**We solve the 1D Poisson equation:**

$$-\nabla^2 u(x) = f(x)$$

With boundary conditions $u(0) = u(1) = 0$ and exact solution $u(x) = \frac{1}{2}x(1-x)$ for $f(x) = 1$.

In [ ]:
class PINNPoisson(nn.Module):
    """PINN for solving the 1D Poisson equation: -u''(x) = f(x)"""
    
    def __init__(self, hidden_dim=128, n_layers=3):
        super().__init__()
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(n_layers):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

def poisson_loss(model, f_true, x_colloc):
    """
    PINN loss = PDE residual + boundary conditions.
    """
    x_colloc = x_colloc.requires_grad_(True)
    u_pred = model(x_colloc)
    
    # Compute second derivative via autograd
    u_x = torch.autograd.grad(u_pred, x_colloc, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_colloc, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    u_xx = u_xx.squeeze(-1)
    
    # PDE residual: -u'' ≈ f
    f_pred = -u_xx
    res_pde = torch.mean((f_pred - f_true) ** 2)
    
    # Boundary conditions: u(0) = u(1) = 0
    x_bc = torch.tensor([[0.0], [1.0]], requires_grad=False)
    u_bc = model(x_bc)
    res_bc = torch.mean(u_bc ** 2)
    
    return res_pde + 100 * res_bc  # Weight for BCs

# Exact forcing function for Poisson
f_exact = lambda x: torch.ones_like(x)  # f(x) = 1
u_exact = lambda x: 0.5 * x * (1 - x)

print("Exact solution: u(x) = 0.5 * x * (1 - x)")
print("Exact forcing: f(x) = 1")

In [ ]:
# Train PINN
pinnet = PINNPoisson(hidden_dim=128, n_layers=4)
pinet_optimizer = torch.optim.Adam(pinnet.parameters(), lr=1e-3)

n_colloc = 200
x_colloc = torch.rand(n_colloc, 1)  # Collocation points

n_epochs_pinn = 500
loss_pinn = []

print(f"Training PINN ({n_epochs_pinn} epochs)...")
for i in range(n_epochs_pinn):
    pinet_optimizer.zero_grad()
    loss = poisson_loss(pinnet, f_exact(x_colloc), x_colloc)
    
    if loss.backward() is None:
        loss.backward()
    
    pinet_optimizer.step()
    loss_pinn.append(loss.item())


In [ ]:
# Evaluate PINN solution
pinet_optimizer.zero_grad()
loss_pinn.append(loss_pinn[-1] if loss_pinn else float('inf'))

In [ ]:
# Evaluate PINN results
x_test = torch.linspace(0, 1, 200).unsqueeze(-1)
u_pred = pinnet(x_test).detach().numpy().flatten()
u_true = u_exact(x_test).numpy().flatten()

err = np.abs(u_pred - u_true)
max_err = np.max(err)
mean_err = np.mean(err)
rel_err = np.max(err) / np.max(np.abs(u_true)) * 100

print(f"\nPINN Poisson solver results:")
print(f"  Max absolute error: {max_err:.2e}")
print(f"  Mean absolute error: {mean_err:.2e}")
print(f"  Max relative error: {rel_err:.2f}%")

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Top plot: solution comparison
axes[0].plot(x_test.numpy(), u_true, 'b-', linewidth=2, label='Exact')
axes[0].plot(x_test.numpy(), u_pred, 'r--', linewidth=2, label='PINN')
axes[0].set_xlabel('x')
axes[0].set_ylabel('u(x)')
axes[0].set_title('Poisson Equation: PINN vs Exact Solution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom plot: error
axes[1].plot(x_test.numpy(), err, 'c-', linewidth=1.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('PINN Reconstruction Error')

## Part 3: Fourier Neural Operator (FNO)

A neural operator learns the **mapping between function spaces**:

$$u = \mathcal{G}(f)$$

Instead of learning a single input-output pair, the FNO learns $f \mapsto u$ for the Poisson equation.

In [ ]:
class FourierNeuralOperator(nn.Module):
    """
    Simplified 1D FNO for learning differential equation solution operators.
    """
    
    def __init__(self, n_modes=16, hidden_dim=64, n_scales=2):
        super().__init__()
        self.n_modes = n_modes  # Fourier modes to keep
        self.n_scales = n_scales
        
        # Input lifting
        self.input_layer = nn.Linear(1, hidden_dim)
        
        # Spectral layers
        self.spectral_layers = nn.ModuleList()
        for _ in range(n_scales):
            self.spectral_layers.append(nn.Linear(hidden_dim * 2 + 1, hidden_dim))
        
        self.act = F.tanh
    
    def forward(self, x, f):
        """
        Args:
            x: (L, 1) grid points
            f: (L, 1) forcing function values at grid points
        """
        L = x.shape[0]
        
        # Compute Fourier coefficients
        f_fft = torch.fft.rfft(f.squeeze(-1)) / L  # (L//2+1,)
        
        # Keep low-frequency modes (simplified)
        # Lift + spectral transform
        u = self.act(self.input_layer(f))  # (L, hidden)
        
        # Spectral layers with Fourier basis
        for layer in self.spectral_layers:
            # Add frequency features
            freqs = torch.arange(1, self.n_modes + 1, device=x.device).float()
            freq_features = torch.cat([
                torch.sin(freqs.unsqueeze(1) * x),
                torch.cos(freqs.unsqueeze(1) * x)
            ], dim=-1)  # (L, 2*n_modes)
            
            combined = torch.cat([u, freq_features, f], dim=-1)
            u = self.act(u + layer(combined))
        
        # Output projection
        return self.act(nn.Linear(hidden_dim, 1)(u))  # (L, 1)

## FNO Training: Learn the Poisson Solver

Generate training data as (forcing function, solution) pairs. The FNO learns the mapping.

In [ ]:
L = 128  # Grid resolution
x_grid = torch.linspace(0, 1, L).unsqueeze(-1)

def generate_fno_training_data(n_samples=200):
    """Generate random forcing functions and their Poisson solutions."""
    X = []  # Forcing functions
    Y = []  # Solutions
    
    for _ in range(n_samples):
        # Random Fourier series for forcing
        f = torch.zeros(L)
        n_modes = np.random.randint(5, 20)
        for m in range(1, n_modes + 1):
            A = np.random.randn()
            B = np.random.randn()
            f += (A * torch.cos(2 * np.pi * m * x_grid.squeeze()) +
                  B * torch.sin(2 * np.pi * m * x_grid.squeeze()))
        f = f.unsqueeze(-1)  # (L, 1)
        
        # Solve Poisson: -u'' = f with Dirichlet BCs using spectral method
        f_fft = torch.fft.rfft(f.squeeze())
        omegas = torch.zeros(L // 2 + 1)
        for m in range(1, L // 2 + 1):
            omegas[m] = (2 * np.pi * m) ** 2
        omegas[0] = 1e-10  # Avoid div-by-zero for DC component
        u_fft = -f_fft / omegas
        u = torch.fft.irfft(u_fft, N=L)
        
        # Enforce BCs
        u = u - (1 - x_grid.squeeze()) * u[0] - x_grid.squeeze() * u[-1]
        u = u.unsqueeze(-1)
        
        X.append(f)
        Y.append(u)
    
    return torch.stack(X), torch.stack(Y)

X_fno, Y_fno = generate_fno_training_data(200)
train_fno = DataLoader(TensorDataset(X_fno[:160], Y_fno[:160]), batch_size=32, shuffle=True)
val_fno = DataLoader(TensorDataset(X_fno[160:], Y_fno[160:]), batch_size=64)

print(f"FNO dataset: {X_fno.shape[0]} training pairs per batch")
print(f"Grid resolution: {L} points")

In [ ]:
# Train FNO
fno_net = FourierNeuralOperator(n_modes=32, hidden_dim=64, n_scales=2)
fno_optimizer = torch.optim.Adam(fno_net.parameters(), lr=5e-4)

n_fno_epochs = 80
loss_fno = []

x_for_op = x_grid.clone()  # Pre-clone grid

print(f"Training FNO ({n_fno_epochs} epochs)...")
for epoch in range(n_fno_epochs):
    total_loss = 0
    n_batches = 0
    
    for batch_idx, (batch_f, batch_u) in enumerate(train_fno):
        batch_f = batch_f.to(device)
        batch_u = batch_u.to(device)
        
        predicted = fno_net(x_for_op.to(device), batch_f)
        loss = F.mse_loss(predicted, batch_u)
        
        fno_optimizer.zero_grad()
        loss.backward()
        fno_optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    avg_loss = total_loss / n_batches
    loss_fno.append(avg_loss)
    
    if epoch % 20 == 0:
        print(f"FNO Epoch {epoch:3d}/{n_fno_epochs} | Loss = {avg_loss:.6f}")

print("FNO training complete!")

In [ ]:
# Evaluate FNO
val_loss = 0
val_count = 0
with torch.no_grad():
    for batch_f, batch_u in val_fno:
        pred = fno_net(x_for_op.to(device), batch_f.to(device))
        val_loss += F.mse_loss(pred, batch_u.to(device)).item()
        val_count += 1
val_loss /= max(val_count, 1)
print(f"Validation MSE: {val_loss:.6f}")

# Train loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(loss_fno, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('FNO Training Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# FNO solution vs ground truth for one sample
test_f = X_fno[160].unsqueeze(0)  # One sample
test_u = Y_fno[160].unsqueeze(0)
fno_pred = fno_net(x_for_op.to(device), test_f.to(device)).squeeze().detach().cpu().numpy()  # Corrected
u_true = test_u.squeeze().numpy()

axes[1].plot(test_f.squeeze().numpy(), 'k-', alpha=0.5, label='Forcing')
axes[1].plot(u_true, 'r-', linewidth=2, label='Ground truth u')
axes[1].plot(fno_pred, 'b--', linewidth=2, label='FNO prediction')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Value')
axes[1].set_title('FNO: Learn the Poisson Solution Operator')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

## Part 4: Architecture Decision Summary

Now we have all three AI components. Here's how they fit together:


# 04 — AI Architecture for Physics Simulation (Continued)

Now we have all three AI components. Here's how they fit together:


| Component | What it learns | When to use |
|-----------|---------------|------------|
| **GNN** | Force $F(r)$ from atomic structure | Molecular simulations where LJ/density varies |
| **PINN** | PDE solution $u(x)$ via physics constraints | Continuum mechanics, diffusion, electrostatics |
| **FNO** | Mapping $f \mapsto u$ (entire function space) | When you need fast inference for many right-hand sides |

**Key insight**: The GNN learns a *local* potential surface. The FNO learns a *globa1* solution operator. PINNs bridge the gap by using physics as a regularizer.

In [ ]:
# Summary table
print("=" * 70)
print(f"{'Component':<15} {'Method':<25} {'Domain':<30}")
print("=" * 70)
print(f"{'GNN':<15} {'Message Passing':<25} {'Atomic scale (Å)':<30})
print(f"{'PINN':<15} {'Physics Constraints':<25} {'Continuum PDEs':<30})
print(f"{'FNO':<15} {'Fourier Spectral':<25} {'Function spaces':<30})
print("=" * 70)

# Save summary
with open('figs/04_architecture_summary.md', 'w') as f:
    f.write("""# AI Architecture Summary
## GNN Force Field
• Message-passing neural network
• Learns LJ-like potentials from data
• Output: per-atom forces (3D) + energy
• Best for: molecular dynamics acceleration

## PINN Poisson Solver
• Physics as regularization term
• Autograd computes PDE residuals
• Output: solution u(x) for -∇²u = f
• Best for: single PDE instances

## Fourier Neural Operator (FNO)
• Learns f → u mapping
• Spectral layers in Fourier domain
• One model → infinite input functions
• Best for: families of PDEs
""")

print("\nSummary saved to figs/04_architecture_summary.md")